# Automated Casting Defect Detection Using a Convolutional Neural Network

## Project Overview
Manufacturing companies inspect products before sending them to customers. Traditionally, visual examination is manual, labor-intensive, and subject to human error or fatigue. In this project, we build an automated **binary image classification system** using TensorFlow/Keras and Convolutional Neural Networks (CNNs) to classify casting product images as:
- **Non-defective (0)** - `ok_front`
- **Defective (1)** - `def_front`


## 1. Import Libraries


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

## 2. Load and Explore the Dataset
Resize images to 224x224, set batch size 32, create training and validation splits (80/20), and configure prefetching.

In [ ]:
train_directory = "../data/train"
test_directory = "../data/test"
image_size = (224, 224)
batch_size = 32
class_names = ["ok_front", "def_front"]

# Load Training Split (80%)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_directory,
    class_names=class_names,
    validation_split=0.20,
    subset="training",
    seed=42,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary"
)

# Load Validation Split (20%)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    train_directory,
    class_names=class_names,
    validation_split=0.20,
    subset="validation",
    seed=42,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary"
)

# Load Test Dataset
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_directory,
    class_names=class_names,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=False
)

# Configure AUTOTUNE Prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

## 3. Data Augmentation Pipeline
Apply mild augmentation (horizontal flip, small rotation, zoom, translation, contrast) to training images only.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.10)
], name="data_augmentation")

## 4. Build the Convolutional Neural Network (CNN)
Architectural design featuring 3 Convolution + Max-Pooling blocks, Global Average Pooling, Dropout, and Sigmoid output.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1.0 / 255.0),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.40),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(1, activation="sigmoid")
])

model.summary()

## 5. Compile and Configure Callbacks
Use Adam optimizer, binary cross-entropy loss, and callbacks: EarlyStopping, ReduceLROnPlateau, and ModelCheckpoint.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

os.makedirs("../models", exist_ok=True)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(filepath="../models/best_casting_defect_model.keras", monitor="val_loss", save_best_only=True)
]

## 6. Train the Model

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=callbacks
)

## 7. Visualize Training Performance
Plot Accuracy and Loss curves to evaluate learning dynamics.

In [ ]:
epochs_range = range(1, len(history.history['accuracy']) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history['accuracy'], label='Training Accuracy')
plt.plot(epochs_range, history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['loss'], label='Training Loss')
plt.plot(epochs_range, history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

## 8. Evaluate on Test Dataset & Threshold Sweep

In [ ]:
test_results = model.evaluate(test_dataset)
print("Test Loss & Metrics:", test_results)

preds = model.predict(test_dataset).flatten()
actuals = np.concatenate([y.numpy().flatten() for x, y in test_dataset]).astype(int)
pred_labels = (preds >= 0.50).astype(int)

print("\nClassification Report:")
print(classification_report(actuals, pred_labels, target_names=class_names))

cm = confusion_matrix(actuals, pred_labels)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 9. Single Image Prediction Utility

In [ ]:
def predict_product(image_path, model, threshold=0.50):
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    arr = tf.keras.utils.img_to_array(img)
    batch = tf.expand_dims(arr, axis=0)
    prob = float(model.predict(batch, verbose=0)[0][0])
    verdict = "Defective" if prob >= threshold else "Non-defective"
    action = "Send for manual inspection" if verdict == "Defective" else "Product may proceed"
    print(f"Image: {image_path}")
    print(f"Prediction: {verdict}")
    print(f"Defect probability: {prob:.2%}")
    print(f"Action: {action}")

predict_product("../sample_images/sample_def_1.jpg", model)